# Model B — Embedding mixup

Cross-entropy with beta(0.4, 0.4) mixup. Kaggle public LB: 0.739.


## Colab setup

1. Place `repro/` inside your Drive project: `BirdCLEF_Project/repro/`.
2. Open any notebook from that Drive folder in Colab (right-click → Open with → Colab).
3. Runtime: **GPU** (T4 or better).
4. Colab secrets: add `KAGGLE_API_TOKEN` (key icon in the left sidebar).
5. `BirdCLEF_Project/` on Drive must also contain:
   - `perch_v2_no_dft.onnx`
   - `embeddings_v2_archive.zip` (training notebooks)
   - `embeddings_v2_TTA_archive.zip` (TTA / final model notebooks)
   - `train.csv` and `sample_submission.csv` (or download in notebook 02)

The first code cell mounts Drive and loads `birdclef` from `BirdCLEF_Project/repro/`.
Embeddings are unzipped to `/content/` for fast GPU training; checkpoints save to `repro/outputs/`.

After execution, download the notebook with outputs and re-upload for the GitHub delivery pass.


In [1]:
from google.colab import drive

import os
import sys
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/BirdCLEF_Project")
REPRO_ROOT = DRIVE_PROJECT / "repro"

if not (REPRO_ROOT / "birdclef").exists():
    raise FileNotFoundError(
        f"Expected repro at {REPRO_ROOT}. Place repro inside BirdCLEF_Project on Drive."
    )

sys.path.insert(0, str(REPRO_ROOT))
os.chdir(REPRO_ROOT)
print(f"Working directory: {REPRO_ROOT}")

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn

from birdclef.colab import colab_prepare_training

repro_root = colab_prepare_training(stage_tta=False)
print(f"Project root: {repro_root}")


Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.4 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df, NUM_CLASSES, _ = prepare_baseline_data()

In [4]:
SAVE_DIR = MODELS_DIR / "mixup_10ep"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 10
mixup_fold_scores = []

for TRAIN_FOLD in range(5):
    best_auc = 0.0
    train_df = df[df["fold"] != TRAIN_FOLD].reset_index(drop=True)
    valid_df = df[df["fold"] == TRAIN_FOLD].reset_index(drop=True)
    train_ds = BirdDataset(train_df, is_tta=False)
    valid_ds = BirdDataset(valid_df, is_tta=False)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=0)
    fold_train_losses = []
    fold_val_losses = []
    fold_val_aucs = []

    print(f"TRAINING FOLD {TRAIN_FOLD}")
    print(f"Training on {len(train_df)} samples, validating on {len(valid_df)} samples")

    # Fresh model/optimizer per fold so each fold is an independent CV estimate.
    seed_everything(42)
    model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            lam = np.random.beta(0.4, 0.4)
            batch_size = inputs.size()[0]
            index = torch.randperm(batch_size).to(device)
            mixed_inputs = lam * inputs + (1 - lam) * inputs[index]
            optimizer.zero_grad()
            outputs = model(mixed_inputs)
            loss = lam * criterion(outputs, labels) + (1 - lam) * criterion(outputs, labels[index])

            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        avg_val_loss = val_loss / len(valid_loader)
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {current_auc:.4f}"
        )
        fold_train_losses.append(avg_train_loss)
        fold_val_losses.append(avg_val_loss)
        fold_val_aucs.append(current_auc)
        if current_auc > best_auc:
            best_auc = current_auc
            save_path = SAVE_DIR / f"best_moe_fold{TRAIN_FOLD}.pth"
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

    onnx_save_path = os.path.join(str(SAVE_DIR), f"bird_moe_fold{TRAIN_FOLD}.onnx")
    plot_and_save_learning_curves(fold_train_losses, fold_val_losses, fold_val_aucs, TRAIN_FOLD, str(SAVE_DIR))
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    dummy_input = torch.randn(1, 1536).to(device)
    with open(os.devnull, "w") as f, redirect_stdout(f), redirect_stderr(f), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name in ("onnx", "onnxruntime", "torch.onnx", "onnxscript"):
            logging.getLogger(name).setLevel(logging.CRITICAL)
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            export_params=True,
            opset_version=15,
            do_constant_folding=True,
            input_names=["embedding"],
            output_names=["logits"],
            dynamic_axes={"embedding": {0: "batch_size"}, "logits": {0: "batch_size"}},
        )
    print(f"Exported ONNX model to {onnx_save_path}")
    mixup_fold_scores.append(best_auc)

print(f"CV score: {np.mean(mixup_fold_scores):.4f} (+/- {np.std(mixup_fold_scores):.4f})")

100%|██████████| 7110/7110 [00:01<00:00, 4026.13it/s]


TRAINING FOLD 0
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.8907 | Val Loss: 0.8218 | Val AUC: 0.9914
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold0.pth
Epoch 02/10 | Train Loss: 1.3760 | Val Loss: 0.7937 | Val AUC: 0.9910
Epoch 03/10 | Train Loss: 1.1693 | Val Loss: 0.8066 | Val AUC: 0.9913
Epoch 04/10 | Train Loss: 1.1411 | Val Loss: 0.8145 | Val AUC: 0.9908
Epoch 05/10 | Train Loss: 1.0967 | Val Loss: 0.8141 | Val AUC: 0.9907
Epoch 06/10 | Train Loss: 1.0375 | Val Loss: 0.8122 | Val AUC: 0.9884
Epoch 07/10 | Train Loss: 1.0001 | Val Loss: 0.8197 | Val AUC: 0.9891
Epoch 08/10 | Train Loss: 1.0340 | Val Loss: 0.8230 | Val AUC: 0.9889
Epoch 09/10 | Train Loss: 1.0538 | Val Loss: 0.8234 | Val AUC: 0.9885
Epoch 10/10 | Train Loss: 1.0085 | Val Loss: 0.8241 | Val AUC: 0.9885
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/learning_curves_fold0.png
E

100%|██████████| 7110/7110 [00:01<00:00, 4059.94it/s]


TRAINING FOLD 1
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.8918 | Val Loss: 0.8369 | Val AUC: 0.9907
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold1.pth
Epoch 02/10 | Train Loss: 1.3793 | Val Loss: 0.7884 | Val AUC: 0.9916
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold1.pth
Epoch 03/10 | Train Loss: 1.1711 | Val Loss: 0.8003 | Val AUC: 0.9913
Epoch 04/10 | Train Loss: 1.1428 | Val Loss: 0.7938 | Val AUC: 0.9898
Epoch 05/10 | Train Loss: 1.0941 | Val Loss: 0.8164 | Val AUC: 0.9915
Epoch 06/10 | Train Loss: 1.0421 | Val Loss: 0.8240 | Val AUC: 0.9910
Epoch 07/10 | Train Loss: 0.9995 | Val Loss: 0.8240 | Val AUC: 0.9905
Epoch 08/10 | Train Loss: 1.0357 | Val Loss: 0.8090 | Val AUC: 0.9914
Epoch 09/10 | Train Loss: 1.0538 | Val Loss: 0.8360 | Val AUC: 0.9880
Epoch 10/10 | Train Loss: 1.0076 | Val Loss: 0.8274 | Val AUC: 0.9911
Learning curves sa

100%|██████████| 7110/7110 [00:02<00:00, 2939.16it/s]


TRAINING FOLD 2
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.9053 | Val Loss: 0.7791 | Val AUC: 0.9909
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold2.pth
Epoch 02/10 | Train Loss: 1.3837 | Val Loss: 0.7496 | Val AUC: 0.9930
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold2.pth
Epoch 03/10 | Train Loss: 1.1770 | Val Loss: 0.7538 | Val AUC: 0.9914
Epoch 04/10 | Train Loss: 1.1395 | Val Loss: 0.7698 | Val AUC: 0.9919
Epoch 05/10 | Train Loss: 1.1035 | Val Loss: 0.7684 | Val AUC: 0.9899
Epoch 06/10 | Train Loss: 1.0396 | Val Loss: 0.7760 | Val AUC: 0.9915
Epoch 07/10 | Train Loss: 1.0053 | Val Loss: 0.7813 | Val AUC: 0.9897
Epoch 08/10 | Train Loss: 1.0353 | Val Loss: 0.7925 | Val AUC: 0.9886
Epoch 09/10 | Train Loss: 1.0560 | Val Loss: 0.7969 | Val AUC: 0.9894
Epoch 10/10 | Train Loss: 1.0080 | Val Loss: 0.7806 | Val AUC: 0.9893
Learning curves sa

100%|██████████| 7110/7110 [00:01<00:00, 4542.36it/s]


TRAINING FOLD 3
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.8991 | Val Loss: 0.7920 | Val AUC: 0.9903
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold3.pth
Epoch 02/10 | Train Loss: 1.3844 | Val Loss: 0.7534 | Val AUC: 0.9920
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold3.pth
Epoch 03/10 | Train Loss: 1.1802 | Val Loss: 0.7442 | Val AUC: 0.9917
Epoch 04/10 | Train Loss: 1.1506 | Val Loss: 0.7533 | Val AUC: 0.9920
Epoch 05/10 | Train Loss: 1.1005 | Val Loss: 0.7616 | Val AUC: 0.9906
Epoch 06/10 | Train Loss: 1.0425 | Val Loss: 0.7723 | Val AUC: 0.9911
Epoch 07/10 | Train Loss: 1.0097 | Val Loss: 0.7655 | Val AUC: 0.9914
Epoch 08/10 | Train Loss: 1.0365 | Val Loss: 0.7641 | Val AUC: 0.9900
Epoch 09/10 | Train Loss: 1.0534 | Val Loss: 0.7988 | Val AUC: 0.9892
Epoch 10/10 | Train Loss: 1.0171 | Val Loss: 0.7664 | Val AUC: 0.9910
Learning curves sa

100%|██████████| 7109/7109 [00:01<00:00, 4180.42it/s]


TRAINING FOLD 4
Training on 28440 samples, validating on 7109 samples
Epoch 01/10 | Train Loss: 1.9095 | Val Loss: 0.7809 | Val AUC: 0.9911
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold4.pth
Epoch 02/10 | Train Loss: 1.3835 | Val Loss: 0.7451 | Val AUC: 0.9931
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/mixup_10ep/best_moe_fold4.pth
Epoch 03/10 | Train Loss: 1.1749 | Val Loss: 0.7469 | Val AUC: 0.9922
Epoch 04/10 | Train Loss: 1.1455 | Val Loss: 0.7511 | Val AUC: 0.9922
Epoch 05/10 | Train Loss: 1.1048 | Val Loss: 0.7833 | Val AUC: 0.9896
Epoch 06/10 | Train Loss: 1.0385 | Val Loss: 0.7727 | Val AUC: 0.9919
Epoch 07/10 | Train Loss: 1.0011 | Val Loss: 0.7605 | Val AUC: 0.9906
Epoch 08/10 | Train Loss: 1.0407 | Val Loss: 0.7788 | Val AUC: 0.9917
Epoch 09/10 | Train Loss: 1.0553 | Val Loss: 0.8002 | Val AUC: 0.9903
Epoch 10/10 | Train Loss: 1.0134 | Val Loss: 0.7814 | Val AUC: 0.9909
Learning curves sa